# 第40课 · 打印声音的成分表——幅度谱 / 相位谱（magnitude / phase spectrum）与频率分辨率实战

**目标**：用实现好的 FFT 分析真实信号，计算频率轴、理解 bin 间距与实信号对称性。

🔗 Aurora 连接：`aurora.audio.stft.magnitude_spectrogram()` 内部在做完全相同的频率轴映射——本节把这个过程拆开让你自己搭一遍。

← **上一课**　[L39 · 从零手写 FFT](L39_fft_implement.ipynb)

> 上节课学习了 **从零手写 FFT**：Cooley-Tukey 递归实现，与 numpy.fft 误差 < 1e-10。  
> 本课将探讨 **频谱分析实战**。

## 本课剧情：频谱像什么？把声音的"成分表"打出来

你喝一杯果汁，能尝出里面有橙子、苹果还是柠檬。人耳听声音也类似——大脑把混合声波分解成不同频率的"成分"，判断出是低音吉他还是高音笛声。

FFT 正是在做这件事：把时域信号变成**频谱（spectrum）**——每个频率的强度一目了然。

但有个细节容易被忽略：FFT 输出的 `X[k]` 里，`k` 只是"第几个频率桶（bin）"，不是 Hz。要把桶号转成 Hz，需要知道：

$$\Delta f = \frac{f_s}{N}$$

- `fₛ`：采样率（Hz）
- `N`：FFT 点数
- `Δf`：相邻两个频率桶之间的间隔（Hz/bin），也叫**频率分辨率（frequency resolution）**

**手算例子**：`fₛ=8000, N=256` → `Δf=31.25 Hz/bin`  
440 Hz 钢琴 A4 音：落在第 `k = round(440/31.25) = 14` 号桶，对应精确频率 `14×31.25 = 437.5 Hz`。

本节任务：实现 `frequency_bins(N, sr)` — 返回单边频率轴，shape `(N//2+1,)`，从 0 Hz 到 Nyquist。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine
from aurora.audio.transforms import (
    fft as aurora_fft,
    next_power_of_two,
    fft_frequencies as aurora_fft_frequencies,
)
# aurora_fft 是 L39 手写的 Cooley-Tukey 实现；要求输入长度为 2 的幂，
# 任意长度信号须先用 next_power_of_two 补零再传入。


## 1. Bin 间距与频率轴

FFT 将 `[0, sr)` Hz 均匀分成 N 份，因此相邻两个 bin 之间的频率间隔为：

```
Δf = sr / N   (Hz/bin)
```

第 k 个 bin 对应的频率是：

```
freqs[k] = k * sr / N
```

换句话说，完整频率轴为 `freqs = np.arange(N) * sr / N`，范围从 0 Hz 一直到 `(N-1)*sr/N ≈ sr`。
N 越大，Δf 越小，频率分辨率越高。

In [ ]:
sr = 8000
for N in [8, 16, 64]:
    delta_f = sr / N
    print(f'N={N:4d}  Δf = {delta_f:.1f} Hz')

## 2. 实信号对称性与奈奎斯特频率（Nyquist frequency）

当输入 `x[n]` 是实数序列时，FFT 满足共轭对称（conjugate symmetry）：

```
|X[k]| = |X[N-k]|
```

这意味着后半段（k > N/2）完全由前半段决定，没有额外信息。
有效的频率范围是 **0 到 sr/2**，对应 bin 下标 0 到 N//2（共 N//2+1 个 bin）。
最高频率 `sr/2` 称为**奈奎斯特频率**——采样定理（Nyquist-Shannon sampling theorem）要求信号不含高于它的分量，否则发生混叠（aliasing）。

In [ ]:
# 验证对称性（使用 L39 手写的 aurora_fft，N=16 = 2^4，无需补零）
sr = 8000
N = 16
rng = np.random.default_rng(42)
x = rng.standard_normal(N)
X = aurora_fft(x)
# 确认 aurora_fft 与 numpy 结果一致
assert np.allclose(aurora_fft(x), np.fft.fft(x)), "aurora_fft 应与 np.fft.fft 完全一致"
print('|X[1]| vs |X[N-1]|:', np.abs(X[1]).round(6), np.abs(X[N-1]).round(6))
print('|X[2]| vs |X[N-2]|:', np.abs(X[2]).round(6), np.abs(X[N-2]).round(6))
print('奈奎斯特频率 =', sr // 2, 'Hz，对应 bin', N // 2)


## 3. 幅度谱（magnitude spectrum）、功率谱（power spectrum）与 dB 谱

对 FFT 系数 `X[k]` 取不同的标量变换，得到三种常用频谱：

```
幅度谱  A[k] = |X[k]|
功率谱  P[k] = |X[k]|²
dB 谱   D[k] = 20 * log10(|X[k]| + eps)
```

dB 谱把乘法关系变成加法，动态范围大的信号（比如音频）在 dB 下更容易分辨细节。
`eps`（通常取 `1e-8`）防止对零取对数。

In [ ]:
sr = 8000
duration = 0.5
x_demo = sine(440, duration, sr)  # 440 Hz 纯音
N_demo = len(x_demo)          # 4000，非 2 的幂
N_fft  = next_power_of_two(N_demo)  # 4096 = 2^12，补零后长度
x_padded = np.zeros(N_fft)
x_padded[:N_demo] = x_demo
X_demo = aurora_fft(x_padded)  # 使用 L39 手写 FFT（aurora.audio.transforms.fft）
assert np.allclose(X_demo, np.fft.fft(x_padded)), "aurora_fft 应与 np.fft.fft 一致"
half = N_fft // 2 + 1
freqs_demo = np.arange(half) * sr / N_fft

A = np.abs(X_demo[:half])
P = A ** 2
D = 20 * np.log10(A + 1e-8)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].plot(freqs_demo, A); axes[0].set_title('幅度谱'); axes[0].set_xlabel('Hz')
axes[1].plot(freqs_demo, P); axes[1].set_title('功率谱'); axes[1].set_xlabel('Hz')
axes[2].plot(freqs_demo, D); axes[2].set_title('dB 谱'); axes[2].set_xlabel('Hz')
plt.tight_layout()
plt.show()


## 3b. 相位谱（phase spectrum）

复数 FFT 系数 `X[k]` 包含两部分信息：

```
X[k] = |X[k]| · e^{jφ[k]}
幅度谱  A[k] = |X[k]|        → 每个频率分量的强度
相位谱  φ[k] = angle(X[k])   → 每个频率分量的初相位（弧度）
```

**实信号的相位反对称性**：

```
φ[k] = −φ[N−k]   （与幅度对称性对应）
```

这意味着相位谱的后半段是前半段的负镜像——与共轭对称性完全一致。

**物理意义**：对于单频正弦 `sin(2π f₀ t + θ₀)`，FFT 在 f₀ 对应 bin 处的相位
就是 `θ₀`（初相位）。相位谱在音频水印、相位声码器（phase vocoder）和
时间拉伸（time-stretching）中扮演核心角色。


In [ ]:
# 相位谱示例：440 Hz + 880 Hz 混合信号（补零到 2 的幂）
sr = 8000
x_phase = sine(440, 0.5, sr) + sine(880, 0.5, sr)
N_p = len(x_phase)
N_fft_p = next_power_of_two(N_p)
x_phase_padded = np.zeros(N_fft_p)
x_phase_padded[:N_p] = x_phase
X_phase = aurora_fft(x_phase_padded)
half_p = N_fft_p // 2 + 1
freqs_p = np.arange(half_p) * sr / N_fft_p

phi = np.angle(X_phase[:half_p])   # φ[k] = arctan2(Im, Re)，单位：弧度

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))
ax1.plot(freqs_p, np.abs(X_phase[:half_p]))
ax1.set_xlim(0, 1200)
ax1.set_title('幅度谱（参考）')
ax1.set_ylabel('幅度')
ax2.plot(freqs_p, phi)
ax2.set_xlim(0, 1200)
ax2.axhline(0, color='k', linewidth=0.5)
ax2.set_title('相位谱 φ[k] = angle(X[k])')
ax2.set_xlabel('频率 (Hz)')
ax2.set_ylabel('相位 (rad)')
plt.tight_layout()
plt.show()

# 验证实信号相位反对称性：φ[k] ≈ −φ[N_fft_p − k]
X_full = aurora_fft(x_phase_padded)
phi_full = np.angle(X_full)
k_test = 5
print(f'φ[{k_test}] = {phi_full[k_test]:.4f}  rad')
print(f'−φ[N−{k_test}] = {-phi_full[N_fft_p - k_test]:.4f}  rad')
print('反对称性成立:', np.isclose(phi_full[k_test], -phi_full[N_fft_p - k_test], atol=1e-10))


## 4. ✏️ 实现 `frequency_bins(N, sr)`

**物理意义**：第 k 个 FFT 系数对应的频率 = `k × sr / N`（Hz）。

**为什么只返回一半？**  
实信号的 FFT 满足共轭对称：`X[N-k] = X[k]*`。单边谱只需 `k = 0..N//2`，共 `N//2+1` 个点，从 0 Hz 到 Nyquist（sr/2）。

**两步实现路线**：

| 步骤 | 代码 | 说明 |
|---|---|---|
| 1 | `freqs_full = np.arange(N) * sr / N` | 完整频率轴，长度 N |
| 2 | `return freqs_full[:N//2 + 1]` | 取前半段（含 Nyquist） |

**验收标准**：
- `frequency_bins(8, 8000)[0] == 0.0`（DC 分量）
- `frequency_bins(8, 8000)[-1] == 4000.0`（Nyquist）
- `frequency_bins(8, 8000)` 长度 = `8//2+1 = 5`

In [ ]:
def frequency_bins(N: int, sr: int) -> np.ndarray:
    """返回实信号 FFT 的频率轴，shape (N//2+1,)，单位 Hz。"""
    # ✏️ TODO: 生成完整频率轴，然后截取前 N//2+1 个
    raise NotImplementedError("TODO: freqs_full = np.arange(N) * sr / N; return freqs_full[:N // 2 + 1]")

In [ ]:
# 数值检查
try:
    result = frequency_bins(8, 8000)
    if result is ... or result is None:
        print('⬜ 请先实现 frequency_bins，再运行此格')
    else:
        assert result[-1] == 4000, f'奈奎斯特应为 4000，得到 {result[-1]}'
        assert len(result) == 5, f'长度应为 5，得到 {len(result)}'
        assert result[0] == 0.0, '第一个 bin 应为 0 Hz'
        assert np.allclose(result, [0., 1000., 2000., 3000., 4000.])
        # 🔗 交叉验证：应与 aurora.audio.transforms.fft_frequencies 数值一致
        ref = aurora_fft_frequencies(8000, 8)
        assert np.allclose(result, ref), (
            f'与 aurora_fft_frequencies 不一致：得到 {result}，期望 {ref}'
        )
        print('✅ frequency_bins 通过所有检查（含与 aurora_fft_frequencies 交叉验证）')
except NotImplementedError:
    print('⬜ frequency_bins 尚未实现，请完成 TODO 后再运行此格')
except TypeError:
    print('⬜ frequency_bins 返回了非数组类型，请完成 TODO')

## 5. 参数实验：双峰定位

生成含 **440 Hz** 和 **880 Hz** 的混合信号，画出单边幅度谱。

**预期现象**：
- 两个尖峰分别精确出现在 440 Hz 和 880 Hz 对应的 bin 附近
- `duration` 越长（N 越大），峰越窄、定位越准（Δf 越小）
- 把 `duration` 改成 `0.01`（N 很小）时，峰会明显变宽

In [ ]:
sr = 8000
duration = 0.5          # ← 改成 0.01 观察频率分辨率变差
x_mix = sine(440, duration, sr) + sine(880, duration, sr)
N = len(x_mix)

# 守护：frequency_bins 未实现时提前退出，避免崩溃
try:
    _fb_check = frequency_bins(N, sr)
except NotImplementedError:
    _fb_check = None
if _fb_check is None:
    print('⬜ 请先实现 frequency_bins，再运行此格')
else:
    # aurora_fft 要求 2 的幂，补零到 next_power_of_two
    N_fft = next_power_of_two(N)
    x_padded = np.zeros(N_fft)
    x_padded[:N] = x_mix
    X_mix = aurora_fft(x_padded)  # L39 手写 FFT
    assert np.allclose(X_mix, np.fft.fft(x_padded)), "aurora_fft 一致性检查"
    freqs = frequency_bins(N_fft, sr)
    amp = np.abs(X_mix[:N_fft // 2 + 1])

    plt.figure(figsize=(9, 3))
    plt.plot(freqs, amp)
    plt.axvline(440, color='r', linestyle='--', label='440 Hz')
    plt.axvline(880, color='g', linestyle='--', label='880 Hz')
    plt.xlabel('频率 (Hz)')
    plt.ylabel('幅度')
    plt.title(f'混合信号幅度谱  (N={N_fft}, Δf={sr/N_fft:.2f} Hz/bin)')
    plt.legend()
    plt.xlim(0, 1200)
    plt.tight_layout()
    plt.show()

    # 找峰值 bin 对应频率
    # 注意：若信号长度非整数周期，旁瓣可能使峰值偏移（谱泄漏现象），
    # 可将 duration 改为 1/sr * round(sr/440) 整数周期来消除泄漏。
    top2_idx = np.argsort(amp)[-2:][::-1]
    print('检测到的两个最强频率:', sorted(freqs[top2_idx].astype(int)), 'Hz')

## 本课收束

`frequency_bins(N, sr)` 把 FFT 的 bin 下标映射成 Hz，输出 shape `(N//2+1,)` 的实数数组，最后一个元素恰好是奈奎斯特频率 `sr/2`。
这条频率轴直接被 `aurora.audio.stft.magnitude_spectrogram()` 用于构造每帧的频率坐标——你在这里写的逻辑就是 STFT 的频率映射层。
幅度谱、功率谱、dB 谱三种形式对应不同的后续处理需求，音频特征提取通常使用 dB 谱。
下一课（L41）将在频谱上叠加 Hann/Hamming 窗函数，完成「分帧→加窗→FFT」全流程。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：频谱分析手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：`sr=8000, N=256`
- Δf = sr/N = ?（Hz/bin）
- 440 Hz 落在第几号 bin？（k = round(440/Δf)）
- 该 bin 对应的精确频率是多少 Hz？

**问 2**：`frequency_bins(8, 8000)` 应返回哪 5 个频率值？  
（形状：(N//2+1,) = (5,)，从 0 到 Nyquist=4000 Hz）

**问 3**：要将 10 Hz 的两个音符区分开，FFT 点数 N 最少需要多少？  
（要求 Δf = sr/N ≤ 10 Hz，sr=16000）

**问 4**：幅度谱 vs 功率谱的关系是什么？  
（提示：功率谱 = 幅度谱²）

**问 5**：为什么频率轴只取 `k=0..N//2`（单边谱），而不是 `k=0..N-1`？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：sr=8000, N=256
sr, N = 8000, 256
delta_f = sr / N
assert delta_f == 31.25
k_440 = round(440 / delta_f)
f_k440 = k_440 * delta_f
assert k_440 == 14
assert f_k440 == 437.5
print(f"Q1 ✅  Δf={delta_f} Hz/bin，440Hz→第{k_440}号bin，精确频率={f_k440} Hz")

# 问2：frequency_bins(8, 8000) = [0, 1000, 2000, 3000, 4000]
expected_bins = np.array([0.0, 1000.0, 2000.0, 3000.0, 4000.0])
try:
    result = frequency_bins(8, 8000)
    assert result.shape == (5,), f"shape 应=(5,)，得到 {result.shape}"
    assert np.allclose(result, expected_bins, atol=1e-10)
    print(f"Q2 ✅  frequency_bins(8, 8000)={result}")
except NotImplementedError:
    print("⬜ Q2：请先实现 frequency_bins()，再运行对答案格")

# 问3：最小 N for Δf≤10 Hz
sr2 = 16000
min_N = int(np.ceil(sr2 / 10))
# 需向上取2的幂（Cooley-Tukey要求）
N_pow2 = 2 ** int(np.ceil(np.log2(min_N)))
assert min_N == 1600
print(f"Q3 ✅  最小 N={min_N}（Δf={sr2/min_N} Hz），取2的幂则 N={N_pow2}")

# 问4：幅度谱 vs 功率谱
X = np.fft.rfft(np.array([1.0, 0.0, -1.0, 0.0]))
mag = np.abs(X)
power = mag ** 2
assert np.allclose(power, mag**2, atol=1e-15)
print(f"Q4 ✅  幅度谱={np.round(mag,4)}，功率谱=幅度谱²={np.round(power,4)}")

# 问5：单边谱 — 实信号共轭对称
x_real = np.random.randn(8)
X_full = np.fft.fft(x_real)
# 验证 X[N-k] = conj(X[k])
for k_val in range(1, 4):
    assert np.isclose(X_full[8-k_val], np.conj(X_full[k_val]), atol=1e-10)
print(f"Q5 ✅  实信号满足 X[N-k]=X[k]*（共轭对称），后半段无新信息，只取 k=0..N/2")
print("\n🎉 频谱分析白板挑战通过！frequency_bins(N,sr)=k*sr/N，k=0..N//2。")

In [ ]:
# ✏️ 本课自评
l40_review = {
    "frequency_resolution":    None,  # 记住 Δf=sr/N（Hz/bin）？True/False
    "frequency_bins_impl":     None,  # frequency_bins 实现并通过断言？True/False
    "magnitude_vs_power":      None,  # 理解幅度谱=|X[k]|，功率谱=|X[k]|²？True/False
    "one_sided_spectrum":      None,  # 理解实信号只需单边谱（共轭对称）？True/False
    "whiteboard_passed":       None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l40_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l40_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L40 全部通关！进入 L41：加窗 FFT 完整流程')

---

→ **下一课**　[L41 · 加窗 FFT 完整流程](L41_fft_full.ipynb)

> 下节课将学习 **加窗 FFT 完整流程**：信号 → 加窗 → FFT → 幅度谱，一条管线跑通。